# Stage 5 Direct-Preservation Precheck

Run the single cell below in a fresh Colab GPU runtime. It does **not** rerun the long scale64 trace-SFT job. It fetches the latest GitHub bootstrap, verifies the direct-preservation precheck markers, sets `HF_TOKEN` from Colab secrets when available, and runs only `STAGE5_CURRENT_A100_TARGET=traced_sft_direct_preservation_precheck`.

This is the cheap gate that decides whether the scale64 checkpoint already preserves base behavior through the loop-1 direct route, or whether the bounded direct-preservation repair sweep is actually needed.

In [ ]:
import base64, json, os, time, urllib.request
from google.colab import userdata

gh = userdata.get("GH_TOKEN") or userdata.get("GITHUB_TOKEN")
assert gh, "Missing GH_TOKEN/GITHUB_TOKEN in Colab secrets."

hf = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_HUB_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf

os.environ["STAGE5_CURRENT_A100_TARGET"] = "traced_sft_direct_preservation_precheck"

url = (
    "https://api.github.com/repos/mshapiro123/recurrent-qwen-svgd/"
    f"contents/colab/CURRENT_A100_BOOTSTRAP_CELL.py?ref=main&t={int(time.time())}"
)
req = urllib.request.Request(
    url,
    headers={
        "Authorization": f"Bearer {gh}",
        "Accept": "application/vnd.github+json",
        "Cache-Control": "no-cache",
    },
)
payload = json.loads(urllib.request.urlopen(req).read().decode("utf-8"))
code = base64.b64decode(payload["content"]).decode("utf-8")

required = [
    "sha_resolved_nested_fetch_v3",
    "traced_sft_direct_preservation_precheck",
    "STAGE5_DIRECT_PRESERVE_PRECHECK_ONLY",
    "direct_route_precheck_needs_training",
]
missing = [marker for marker in required if marker not in code]
assert not missing, f"Fetched bootstrap is stale or incomplete: {missing}"

print("Fetched bootstrap sha:", payload.get("sha"))
exec(compile(code, "colab/CURRENT_A100_BOOTSTRAP_CELL.py", "exec"))
